# Road Following Live (Pure ONNX + Stanley + ROS Subscriber + Interactive Sliders UI)

This notebook subscribes to the **ROS Camera Topic** (`/csi_cam_0/image_raw`), running ONNX model inference on CUDA GPU and **Stanley Control** on physical JetRacer (Zero PyTorch dependency).
It provides an **Interactive `ipywidgets` UI** to adjust Stanley parameters (gain $k$, throttle, brake gain, steering bias, alpha) in real time while displaying the live camera feed and recording video to `output_drive.mp4`.

### 1. Setup Environment & Load ONNX Model on CUDA GPU

In [1]:
import os
import sys
import ctypes
from pathlib import Path

# Pre-load CUDA C++ libraries into RTLD_GLOBAL symbol table so ONNX Runtime dlopen succeeds!
cuda_lib64 = "/usr/local/cuda/lib64"
for lib_name in ["libcudart.so", "libcudnn.so", "libcuda.so"]:
    lib_path = os.path.join(cuda_lib64, lib_name)
    if os.path.exists(lib_path):
        try:
            ctypes.CDLL(lib_path, mode=ctypes.RTLD_GLOBAL)
        except Exception:
            pass

# Add parent directory to sys.path
parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))

import onnxruntime as ort
try:
    from jetracer.utils import preprocess_onnx, bgr8_to_jpeg
except ImportError:
    from utils import preprocess_onnx, bgr8_to_jpeg

# Locate ONNX model file
model_path = os.path.join(Path.cwd(), "road_following_model.onnx")
if not os.path.exists(model_path):
    model_path = os.path.join(parent_dir, "notebooks", "road_following_model.onnx")

if not os.path.exists(model_path):
    print(f"[!] ERROR: ONNX model file '{model_path}' not found!")
else:
    print(f"[*] Loading ONNX model from: {model_path}")

# Configure ONNX Providers
available_providers = ort.get_available_providers()
print(f"[*] Available ONNX Providers: {available_providers}")

providers = []
if 'CUDAExecutionProvider' in available_providers:
    providers.append('CUDAExecutionProvider')
if 'TensorrtExecutionProvider' in available_providers:
    providers.append('TensorrtExecutionProvider')
providers.append('CPUExecutionProvider')

try:
    session = ort.InferenceSession(model_path, providers=providers)
    print(f"[+] Successfully loaded ONNX Session Providers: {session.get_providers()}")
except Exception as err:
    print(f"[!] ONNX Init Exception: {err}")
    session = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])
    print(f"[*] Fallback ONNX Session Providers: {session.get_providers()}")

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name


[*] Loading ONNX model from: /home/jetson/JetRacer_AI/jetracer/notebooks/road_following_model.onnx
[*] Available ONNX Providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
[+] Successfully loaded ONNX Session Providers: ['CUDAExecutionProvider', 'TensorrtExecutionProvider', 'CPUExecutionProvider']


### 2. Initialize ROS Node & JetRacer Hardware (`NvidiaRacecar`)

In [2]:
import rospy
from sensor_msgs.msg import Image as ROSImage
from jetracer.nvidia_racecar import NvidiaRacecar
try:
    from jetracer.Controller import StanleyController
    from jetracer.Runner import JetRacerROSOnnxRunner
except ImportError:
    from Controller import StanleyController
    from Runner import JetRacerROSOnnxRunner

# 1. Initialize ROS Node
try:
    rospy.init_node('road_following_live_notebook', anonymous=True, disable_signals=True)
    print("[+] ROS Node initialized successfully!")
except Exception as e:
    print(f"[*] ROS Node notice: {e}")

# 2. Hardware & Controller Setup
car = NvidiaRacecar()
stanley = StanleyController()
stanley.reset()
print("[+] JetRacer hardware and Stanley Controller initialized.")


WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


[+] ROS Node initialized successfully!
[+] JetRacer hardware and Stanley Controller initialized.


### 3. Interactive Sliders UI & ROS Subscriber Setup

In [3]:
import cv2
import base64
import ipywidgets
from IPython.display import display

# Unregister previous ROS subscriber if active
if 'ros_sub' in globals() and ros_sub is not None:
    try:
        ros_sub.unregister()
    except Exception:
        pass

# Interactive Parameter Sliders
k_slider        = ipywidgets.FloatSlider(min=0.0, max=10.0, step=0.1, value=2.5,  description='Gain (k)', layout=ipywidgets.Layout(width='340px'))
throttle_slider = ipywidgets.FloatSlider(min=0.0, max=1.0,  step=0.01, value=0.20, description='Throttle', layout=ipywidgets.Layout(width='340px'))
brake_slider    = ipywidgets.FloatSlider(min=0.0, max=1.0,  step=0.01, value=0.10, description='Brake Gain', layout=ipywidgets.Layout(width='340px'))
bias_slider     = ipywidgets.FloatSlider(min=-1.0, max=1.0, step=0.01, value=0.0,  description='Steer Bias', layout=ipywidgets.Layout(width='340px'))
alpha_slider    = ipywidgets.FloatSlider(min=0.0, max=1.0,  step=0.05, value=0.4,  description='Alpha', layout=ipywidgets.Layout(width='340px'))

# State Toggle & Display Widgets
state_widget = ipywidgets.ToggleButtons(options=['STOP', 'RUN'], description='Drive State', value='STOP')
camera_html_widget = ipywidgets.HTML(
    value="<p><b>Waiting for ROS Camera Topic...</b></p>",
    layout=ipywidgets.Layout(width='240px', height='240px')
)

video_output_path = os.path.join(Path.cwd(), "output_drive.mp4")

# Frame callback for live HTML UI update
def on_live_frame(cv_image, raw_x, raw_y, smoothed_x, steering, dyn_throttle):
    h, w = cv_image.shape[:2]
    px = int(w * (smoothed_x / 2.0 + 0.5))
    py = int(h * (raw_y / 2.0 + 0.5)) if raw_y != 0.0 else int(h * 0.5)

    prediction = cv_image.copy()
    cv2.circle(prediction, (px, py), 8, (0, 255, 0), -1)
    cv2.putText(prediction, f"Steer:{steering:+.2f} Thr:{dyn_throttle:.2f}", (10, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    jpeg_bytes = bgr8_to_jpeg(prediction)
    b64 = base64.b64encode(jpeg_bytes).decode('utf-8')
    
    html_str = f'''
    <div style="font-family: monospace; background: #1e1e1e; color: #00ff00; padding: 10px; border-radius: 8px; display: inline-block;">
        <h5 style="margin:0 0 4px 0; color: #ffffff;">JetRacer ONNX Live Stream</h5>
        <p style="margin:2px 0; font-size:11px;"><b>Target X:</b> {raw_x:+.2f} | <b>Smooth X:</b> {smoothed_x:+.2f}</p>
        <p style="margin:2px 0; font-size:11px;"><b>Steering:</b> {steering:+.2f} | <b>Throttle:</b> {dyn_throttle:.2f}</p>
        <img src="data:image/jpeg;base64,{b64}" style="width:224px; height:224px; border:2px solid #00ff00; border-radius:4px; margin-top:4px;" />
    </div>
    '''
    camera_html_widget.value = html_str

# Runner setup using dynamic slider reading lambdas
runner = JetRacerROSOnnxRunner(
    session=session,
    input_name=input_name,
    output_name=output_name,
    car=car,
    stanley=stanley,
    k=lambda: k_slider.value,
    throttle=lambda: throttle_slider.value,
    brake_gain=lambda: brake_slider.value,
    bias=lambda: bias_slider.value,
    alpha=lambda: alpha_slider.value,
    video_path=video_output_path,
    video_fps=20.0,
    on_frame=on_live_frame
)

runner.running = False  # Start paused

def on_state_change(change):
    if change['new'] == 'RUN':
        runner.running = True
        stanley.reset()
        print("[+] Autonomous Driving ACTIVE!")
    else:
        runner.stop()
        print("[*] Autonomous Driving STOPPED.")

state_widget.observe(on_state_change, names='value')

topic_name = "/csi_cam_0/image_raw"
ros_sub = rospy.Subscriber(topic_name, ROSImage, runner.image_callback, queue_size=1, buff_size=2**24)

# Assemble Complete Interactive UI Layout
controls_box = ipywidgets.VBox([
    state_widget,
    k_slider,
    throttle_slider,
    brake_slider,
    bias_slider,
    alpha_slider
])

live_ui_widget = ipywidgets.HBox([camera_html_widget, controls_box])
display(live_ui_widget)

print(f"[*] Subscribed to ROS Camera Topic: {topic_name}")
print(f"[*] Video recording configured -> {video_output_path}")


[*] Subscribed to ROS Camera Topic: /csi_cam_0/image_raw
[*] Video recording configured -> /home/jetson/JetRacer_AI/jetracer/notebooks/output_drive.mp4


### 4. Emergency Stop Cell

In [ ]:
# Emergency Stop Cell
state_widget.value = 'STOP'
runner.stop()
